In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import tifffile as tiff
os.chdir('/zhome/71/c/146676/reconstructors_for_hpc')
sys.path.append('/zhome/71/c/146676/reconstructors_for_hpc/x_ray')
from reconstructor import load_volume_from_tiffs, reconstruct_fbp_batch
from postprocessor import stitch_reconstructions
from preprocessor import load_config
from multiprocessing import Pool
import random

In [ ]:
cfg = load_config("reconstruction_settings_1.yaml")


In [ ]:
data = stitch_reconstructions(cfg)

In [ ]:
N_neutron = 1788
paths_NA = [f"/dtu-compute/msaca/sliceA_neutron_psi/output/tv_recon/slice_tv_{i:04d}.tiff" for i in range(N_neutron)]

def loader(path):
    return tiff.imread(path)[:1650,:1650].astype(np.float32)

def load_stack(paths, nproc=8):
    with Pool(processes=nproc) as pool:
        slices = pool.map(loader, paths)
    return np.stack(slices, axis=0)

recon_NA = load_stack(paths_NA, nproc=16)
recon_NA[:200] = 0
recon_NA = np.pad(recon_NA, 
                    pad_width=((400, 0),   # no pad along axis 0
                               (0, 0),   # no pad along axis 1
                               (0, 0)),  # pad 100 before & after along axis 2
                    mode='constant', constant_values=0)

recon_NA = recon_NA[:-300,:,100:-100]

In [ ]:
# start [300, 94, 94, 94, 94] # [693, 694, 694, 694, 788]

end_slice_orig = 693-300+694-94+694-94+694-94+788-90
data_xray_orig = data[:end_slice_orig:2,::2,::2]
data_xray_orig = np.pad(data_xray_orig, 
                    pad_width=((0, 0),   # no pad along axis 0
                               (0, 0),   # no pad along axis 1
                               (100, 100)),  # pad 100 before & after along axis 2
                    mode='constant', constant_values=0)



In [ ]:

#ata1 = data.copy()
#data[end_slice_orig:] = 0

In [ ]:
#data = data1

In [ ]:
import SimpleITK as sitk
fixed = sitk.GetImageFromArray(recon_NA)
moving_small = sitk.GetImageFromArray(data_xray_orig)
moving_big = sitk.GetImageFromArray(data)


f_size = fixed.GetSize()
m_size = moving_small.GetSize()
max_f = max(f_size)
max_m = max(m_size)
max_size = max(max_f,max_m)



fixed.SetSpacing((1.8/max_f,1.8/max_f,1.8/max_f))
moving_small.SetSpacing((1.5/max_m,1.5/max_m,1.5/max_m))
moving_big.SetSpacing((0.75/max_m,0.75/max_m,0.75/max_m))


size_fixed = fixed.GetSize()
size_moving = moving_small.GetSize()
size_moving_big = moving_big.GetSize()

spacing_fixed = fixed.GetSpacing()
spacing_moving = moving_small.GetSpacing()
spacing_moving_big = moving_big.GetSpacing()

# Compute the new origin (shift it to -N/2)

# spatial size is roughly 1.5 cubed
# spacing is roughly 1.5/N_pixels
# Origin 
new_origin_fixed = [-0.5 * (size_fixed[i] - 1) * spacing_fixed[i] for i in range(len(size_fixed))]
new_origin_moving = [-0.5 * (size_moving[i] - 1) * spacing_moving[i] for i in range(len(size_moving))]
new_origin_moving_big = [-0.5 * (size_moving_big[i] - 1) * spacing_moving_big[i] for i in range(len(size_moving_big))]

# Set the origin to be the new center (-N/2)
fixed.SetOrigin(new_origin_fixed)
moving_small.SetOrigin(new_origin_moving)

moving_big.SetOrigin(new_origin_moving_big)


In [ ]:
# fixed = coarse (e.g. 2 µm voxels)
size = np.array(fixed.GetSize())
spacing = np.array(fixed.GetSpacing())

# Double resolution → double size, half spacing
new_size = (size * 2).astype(int).tolist()
new_spacing = (spacing / 2).tolist()

# Create upsampled version of fixed
u_fixed = sitk.Resample(
    fixed,
    new_size,
    sitk.Transform(),       # identity
    sitk.sitkLinear,
    fixed.GetOrigin(),
    new_spacing,
    fixed.GetDirection(),
    0.0,
    fixed.GetPixelID()
)

In [ ]:
transform = sitk.ReadTransform('/zhome/71/c/146676/main/Transformations/transformation_XA_to_NA.tfm')


# Define reference grid = fixed image, but using moving_small spacing
reference = sitk.Image(u_fixed.GetSize(), sitk.sitkFloat32)
reference.SetOrigin(u_fixed.GetOrigin())
reference.SetDirection(u_fixed.GetDirection())
reference.SetSpacing(moving_big.GetSpacing())  # use moving_small resolution


resampled_moving_big = sitk.Resample(
    moving_big,                 # input image (fine resolution)
    u_fixed,                  # reference image -> defines resolution/grid
    transform,              # your similarity3D transform
    sitk.sitkLinear,        # interpolator
    0.0,                    # default pixel value for out-of-bounds
    moving_big.GetPixelID()     # keep same pixel type as moving
)

# # Resample fixed with identity transform
# resampled_fixed = sitk.Resample(
#     fixed,
#     reference,
#     sitk.Transform(),      # identity
#     sitk.sitkLinear,
#     0.0,
#     sitk.sitkFloat32
# )

# # Resample moving_small with the registration transform
# resampled_moving_small = sitk.Resample(
#     moving_small,
#     reference,
#     transform,             # registration transform
#     sitk.sitkLinear,
#     0.0,
#     sitk.sitkFloat32
# )





In [ ]:
print(resampled_moving_big.GetSize())
print(u_fixed.GetSize())

In [ ]:
class Registrator():

    def _downsample_image(self, input_image, factor=10,smooth=False):
        """
        Downsamples a 3D volume by reducing its resolution by a specified factor in each dimension,
        effectively averaging values over blocks (e.g., 2x2x2 cubes when factor=2).

        Args:
            input_image (sitk.Image): The input SimpleITK image to be downsampled.
            factor (int, optional): The downsampling factor. Defaults to 2.
                - A factor of 2 will halve the size of the image in each dimension.
                - The new spacing will be adjusted accordingly.

        Returns:
            sitk.Image: The downsampled SimpleITK image with reduced resolution.

        Function Steps:
            1. Retrieve the original size and spacing of the input image.
            2. Compute the new size by dividing each dimension of the image size by the factor.
            3. Compute the new spacing by multiplying the original spacing by the factor.
            4. Use `sitk.Resample` to resample the image:
                - Use an identity transform to retain the original alignment.
                - Apply `sitk.sitkBSpline` interpolation to achieve smooth averaging.
            5. Return the downsampled image.

        Example:
            # Load a 3D image
            input_image = sitk.ReadImage('large_image.nii')
            
            # Downsample the image to half the size in each dimension
            downsampled_image = downsample_image(input_image, factor=2)
            
            # Save the downsampled image
            sitk.WriteImage(downsampled_image, 'downsampled_image.nii')
        """
        if smooth:
            mean_filter = sitk.MeanImageFilter()
            mean_filter.SetRadius(smooth)  # Adjust the radius as needed
            image = mean_filter.Execute(input_image)
        else:
            image = input_image


        # Get the original size and spacing of the image
        size = image.GetSize()
        spacing = image.GetSpacing()

        # Calculate the new size (downsampling by factor)
        new_size = [int(size[0] / factor), int(size[1] / factor), int(size[2] / factor)]
        
        # Calculate the new spacing (enlarging the spacing to match the downsampled size)
        new_spacing = [s * factor for s in spacing]
        
        # Set the resampling transform (identity transform)
        transform = sitk.Transform(3, sitk.sitkIdentity)

        # Perform the resampling (using average interpolation for downsampling)
        downsampled_image = sitk.Resample(image,
                                        new_size,
                                        transform,
                                        sitk.sitkLinear,  # BSpline interpolation is good for downsampling
                                        image.GetOrigin(),
                                        new_spacing,
                                        image.GetDirection(),
                                        0)  # 0 is the background value for the resampling
        
        return downsampled_image


    def choose_registration_images(self, smoothing_i, shrinking_i, smooth_fixed, smooth_moving):
        if (shrinking_i is not None) and smooth_fixed:
            if str(smoothing_i) in self.fixed_smooth:
                print('Using precalculated smoothed fixed image with shrinking')
                fixed_d = self._downsample_image(self.fixed_smooth[str(smoothing_i)], factor=shrinking_i,smooth = 0)
            else:
                print('Calculating smoothed fixed image with shrinking')
                fixed_d = self._downsample_image(self.fixed, factor=shrinking_i,smooth=smoothing_i)

        if (shrinking_i is not None) and smooth_moving:
            if str(smoothing_i) in self.moving_smooth:
                print('Using precalculated smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving_smooth[str(smoothing_i)], factor=shrinking_i,smooth = 0)
            else:
                print('Calculating smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving, factor=shrinking_i,smooth=smoothing_i)


        if (shrinking_i is not None) and (not smooth_fixed):
            print('Using non-smoothed fixed image with shrinking')
            fixed_d = self._downsample_image(self.fixed, factor=shrinking_i,smooth=0)

        if (shrinking_i is not None) and (not smooth_moving):
                print('Using non-smoothed moving image with shrinking')
                moving_d = self._downsample_image(self.moving, factor=shrinking_i,smooth=0)

        if (shrinking_i is None):
            print('Using non-smoothed moving image without shrinking')
            fixed_d = self.fixed
            moving_d = self.moving

        return fixed_d, moving_d

    def register(self, learning_rate = 0.1, sampling_percentage = 0.1,
                    max_iter = 50,
                    metric_type = 'ms', optimizer_type = 'gd',
                    smoothing = 0,
                    shrinking = 1, thresholds=None, histogram_bins = 50,
                    smooth_fixed = False, smooth_moving = False,mask = None,
                    mask_box = None, unit = 'index'):
        """
        Perform the image registration using the provided transformation parameters.
        
        Args:
            transform_params: The transformation parameters (e.g., translation, rotation).
            metric_type (str): The metric used for the registration (e.g., 'MeanSquares', 'Mattes').
            optimizer_type (str): The optimizer used for the registration (e.g., 'GradientDescent').
        
        Returns:
            SimpleITK.Transform: The resulting transformation after registration.
        """


        initial_transform = sitk.Similarity3DTransform()

        # Explicitly set the matrix to the identity matrix
        initial_transform.SetMatrix([1.0, 0.0, 0.0,
                                    0.0, 1.0, 0.0,
                                    0.0, 0.0, 1.0])

        self.metric_values = []
        self.iterations = []
        # Set the translation to zero
        initial_transform.SetTranslation([0.0, 0.0, 0.0])

        # Set the scale to 1.0
        initial_transform.SetScale(1.0)

        registration = sitk.ImageRegistrationMethod()

        self.first_metric_value = None
        self.second_metric_value = None

        registration.SetInitialTransform(initial_transform)

        fixed_d, moving_d = self.choose_registration_images(smoothing, shrinking, smooth_fixed, smooth_moving)


        if metric_type == 'mmi':
            registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=histogram_bins)
        elif metric_type == 'ms' or metric_type == 'ls':
            registration.SetMetricAsMeanSquares()
            fixed_d = sitk.BinaryThreshold(fixed_d, lowerThreshold=thresholds[0], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
            moving_d = sitk.BinaryThreshold(moving_d, lowerThreshold=thresholds[1], upperThreshold=float("inf"), insideValue=1, outsideValue=0)
            fixed_d =sitk.Cast(fixed_d, sitk.sitkFloat32)
            moving_d =sitk.Cast(moving_d, sitk.sitkFloat32)

        else:
            raise ValueError('Specify metric type: Either "mmi" or "ms"/"ls"')

        if mask == 'fixed':
            mask = self.get_registration_mask(mask_box, which_image = 'fixed', unit = unit)
            registration.SetMetricFixedMask(mask)
        elif mask == 'moving':
            mask = self.get_registration_mask(mask_box, which_image = 'moving', unit = unit)
            registration.SetMetricFixedMask(mask)

        registration.SetMetricSamplingStrategy(registration.RANDOM)
        registration.SetMetricSamplingPercentage(sampling_percentage)

        if optimizer_type == 'gd':
            registration.SetOptimizerAsGradientDescent(
            learningRate=learning_rate,
            numberOfIterations=max_iter,
            convergenceMinimumValue=-1e-16,
            convergenceWindowSize=1000
        )


        else:
            raise ValueError('Specify optimization algorithm: Only "gd" currently supported')
                            
        #registration.SetOptimizerScalesFromPhysicalShift(smallParameterVariation = 0.01)
        registration.SetInterpolator(sitk.sitkLinear)


        self.metric_values = []
        self.iterations = []
            
        registration.Execute(fixed_d, moving_d)
        transform = registration.GetInitialTransform()
        initial_transform = sitk.Similarity3DTransform(transform)

        return transform

In [ ]:
reg = Registrator()
reg.fixed = fixed
reg.moving = resampled_moving_big
threshold = [0.025,0.8]
# transform_registration= reg.register(learning_rate = 1, sampling_percentage = 1.0, max_iter = 1000,
#                                         metric_type = 'ms', optimizer_type = 'gd',
#                                         smoothing = 0, shrinking = 9, thresholds=threshold,
#                                         smooth_fixed = False, smooth_moving = False)

resampled_moving_big2 = sitk.Resample(
    resampled_moving_big,                 # input image (fine resolution)
    u_fixed,                  # reference image -> defines resolution/grid
    transform_registration,              # your similarity3D transform
    sitk.sitkLinear,        # interpolator
    0.0,                    # default pixel value for out-of-bounds
    moving_small.GetPixelID()     # keep same pixel type as moving
)



In [ ]:
px = (sitk.GetArrayFromImage(resampled_moving_big)[:,300,:]>0.8)*1.0
px2 =(sitk.GetArrayFromImage(resampled_moving_big2)[:,300,:]>0.6)*1.0
pn = (sitk.GetArrayFromImage(u_fixed)[:,300,:]>0.025)*1.0
pn_small = (sitk.GetArrayFromImage(fixed)[:,150,:]>0.025)*1.0

In [ ]:
# neutron pixel size is 6.2um

In [ ]:
plt.imshow(pn_small - px2[::2,::2])
plt.show()

In [ ]:
plt.imshow(px)
plt.colorbar()
plt.show()
plt.imshow(pn)
plt.colorbar()
plt.show()
plt.imshow(px-pn)
plt.show()
plt.imshow(pn-px2)
plt.show()

In [ ]:
neutron = (sitk.GetArrayFromImage(fixed)/100*1e5/6.2).astype(np.float32) ##### factor 10 wrong!
x_ray = sitk.GetArrayFromImage(resampled_moving_big2).astype(np.float32) ##### this actually needs to be the image where the bottom has not been set to 0

In [ ]:
np.prod(x_ray.shape)*4/1e+9 + np.prod(x_ray[::2,::2,::2].shape)*4/1e+9 + np.prod(x_ray[::4,::4,::4].shape)*4/1e+9 + np.prod(x_ray[::8,::8,::8].shape)*4/1e+9 + np.prod(neutron.shape)*4/1e+9 + np.prod(neutron[::2,::2,::2].shape)*4/1e+9 + np.prod(neutron[::4,::4,::4].shape)*4/1e+9

In [ ]:
import numpy as np
import zarr
from ome_zarr.writer import write_multiscale
from skimage.transform import downscale_local_mean

def build_pyramid(array, levels):
    pyr = [array.astype(np.float32)]
    for i in range(1, levels):
        factor = 2
        # downscale along z,y,x
        down = downscale_local_mean(pyr[-1], (factor, factor, factor))
        pyr.append(down.astype(np.float32))
    return pyr

# # Example arrays
# # x_ray = np.load("xray.npy")        # shape (Z,Y,X)
# # neutron = np.load("neutron.npy")   # shape (Z,Y,X)

# # Make pyramids
pyr_xray = build_pyramid(x_ray, levels=4)     # full, 2×, 4×, 8×
pyr_neutron = build_pyramid(neutron, levels=3) # full, 2×, 4×

# Save to ome-zarr
# Save to ome-zarr (overwrites if existing)
root = zarr.open_group("/dtu-compute/msaca/sliceA_xray_pc/sliceA_reconstructions.ome.zarr", mode="w")

# Save neutron
grp_neutron = root.create_group("neutron")
write_multiscale(
    pyr_neutron,
    group=grp_neutron,
    axes=[
        {"name": "z", "type": "space", "unit": "micrometer"},
        {"name": "y", "type": "space", "unit": "micrometer"},
        {"name": "x", "type": "space", "unit": "micrometer"},
    ],
    coordinate_transformations=[
    [{"type": "scale", "scale": [6.2, 6.2, 6.2]}],   # level 0 (full res)
    [{"type": "scale", "scale": [12.4, 12.4, 12.4]}], # level 1 (2× downsampled)
    [{"type": "scale", "scale": [24.8, 24.8, 24.8]}], # level 2 (4× downsampled)
]
)

# Save x-ray
#del root["xray"]
grp_xray = root.create_group("xray")
write_multiscale(
    pyr_xray,
    group=grp_xray,
    axes=[
        {"name": "z", "type": "space", "unit": "micrometer"},
        {"name": "y", "type": "space", "unit": "micrometer"},
        {"name": "x", "type": "space", "unit": "micrometer"},
    ],
    coordinate_transformations=[
        [{"type": "scale", "scale": [3.1, 3.1, 3.1]}],     # full res
        [{"type": "scale", "scale": [6.2, 6.2, 6.2]}],     # 2× down
        [{"type": "scale", "scale": [12.4, 12.4, 12.4]}],  # 4× down
        [{"type": "scale", "scale": [24.8, 24.8, 24.8]}],  # 8× down
    ]
)

In [ ]:
import zarr
import SimpleITK as sitk
import numpy as np

# --------------------------
# Step 1: open the OME-Zarr datasets
# --------------------------
root = zarr.open_group("/dtu-compute/msaca/sliceA_xray_pc/sliceA_reconstructions.ome.zarr", mode="r")
xray = root["xray/1"]      # full resolution
neutron = root["neutron/0"]

# --------------------------
# Step 2: helper to extract subvolumes
# --------------------------
def extract_subvolume(zarr_array, start, size):
    """
    Extract subvolume from a zarr array.
    start: (z, y, x) starting index
    size: (dz, dy, dx) size of subvolume
    """
    z0, y0, x0 = start
    dz, dy, dx = size
    subarr = zarr_array[z0:z0+dz, y0:y0+dy, x0:x0+dx]
    return sitk.GetImageFromArray(subarr.astype(np.float32))

# --------------------------
# Step 3: select three matching subvolumes
# --------------------------
# Example: manually chosen coordinates (adjust!)
levelx = 1
leveln = 1
coords_xray = [
    (1300//levelx, 500//levelx, 500//levelx),   # z,y,x start
    (300//levelx, 1000//levelx, 1200//levelx),
    (700//levelx, 2000//levelx, 2500//levelx),
]
coords_neutron = [
    (1300//leveln, 500//leveln, 500//leveln),    # corresponding neutron subvolumes
    (300//leveln, 1000//leveln, 1200//leveln),
    (700//leveln, 2000//leveln, 2500//leveln),
]
size = (128, 128, 128)  # use 128³ subvolumes

xray_subs = [extract_subvolume(xray, s, size) for s in coords_xray]
neutron_subs = [extract_subvolume(neutron, s, size) for s in coords_neutron]

# --------------------------
# Step 4: registration framework
# --------------------------
def register_pair(fixed, moving, initial=None):
    """Register one pair of SITK images and return a transform."""
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMeanSquares()
    reg.SetOptimizerAsRegularStepGradientDescent(
        learningRate=2.0,
        minStep=1e-4,
        numberOfIterations=200,
        gradientMagnitudeTolerance=1e-6
    )
    reg.SetInterpolator(sitk.sitkLinear)
    if initial is None:
        initial = sitk.Similarity3DTransform()
    reg.SetInitialTransform(initial, inPlace=False)
    final = reg.Execute(fixed, moving)
    return final

# --------------------------
# Step 5: run on three subvolume pairs
# --------------------------
transforms = []
for f, m in zip(neutron_subs, xray_subs):
    t = register_pair(f, m)
    transforms.append(t)

# --------------------------
# Step 6: combine transforms
# --------------------------
# For simplicity: average parameters of Similarity3DTransform
def average_similarity(transforms):
    params = np.array([t.GetParameters() for t in transforms])
    mean_params = params.mean(axis=0)
    T = sitk.Similarity3DTransform()
    T.SetParameters(tuple(mean_params))
    return T

final_transform = average_similarity(transforms)

# --------------------------
# Step 7: apply to full x-ray volume
# --------------------------
resampled_xray = sitk.Resample(
    sitk.GetImageFromArray(xray[:].astype(np.float32)),  # full xray into memory
    sitk.GetImageFromArray(neutron[:].astype(np.float32)), # neutron defines grid
    final_transform,
    sitk.sitkLinear,
    0.0,
    sitk.sitkFloat32
)